# 5 — Full Pipeline Integration · Volumetric 3D Arena

Mirror of notebook 4 (the flat 2-D baseline) on the **free-flying volumetric 3-D arena** — the *primary* test of **H1/H2** in the 3×2 design (three environments × {true, Bayesian} plane modes).

**What this run asks.** With the Bingham filter estimating the motion-plane normal n̂ (`plane_mode="bayesian"`), does the integrated Bingham + T³ QAN system reproduce the empirical signature of volumetric grids: **local order but no global lattice** (Ginosar et al., 2021; Grieves et al., 2021)?

**Read the result the opposite way to notebook 4.** In the 2-D baseline a *high* hexagonal grid score is the pass condition. Here, **H1 predicts the global structure scores (χ_FCC, χ_HCP, χ_COL) sit near the RND baseline**, while local order survives in the radial autocorrelation (MRA), inter-field distances, and first-ring metrics. A low global χ is a *success*, not a failure.

**Two changes are not cosmetic** (details below the run): scoring moves from 2-D hexagonal gridness → the 3-D structure-score pipeline (`score_run` + prototype references), because an XY-projected gridness in a volume is exactly the metric Grieves et al. found at chance; and the run is saved with `record=True` so the 3-D scorer has `S_tot_buffer` to read.

In [ ]:
!pip install torch numpy matplotlib
!pip install git+https://github.com/jacobsvennevik/MADE.git

In [ ]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

sys.path.insert(0, '/workspace/CAN_Path_Integration_3D_model')

from network.visualize3D import visualize_trajectory_projections, plot_pi_error
from experiments.arena_3d import Arena3DExperiment            # 3-D volumetric walk (was Arena2DExperiment)
from config import RunConfig, NetworkConfig, ExperimentConfig
from metrics import wrapped_angle_diff

# --- 3-D structure-score pipeline (replaces the 2-D score_2d / score_2d_from_map) ---
# Entry points per the codebase technical reference (analysis layer). Verify the exact
# argument / return-key names against your scoring.py + prototypes.py before trusting numbers.
from analysis.scoring import scoring_input_from_npz, score_run, activity_rate_map
from analysis.prototypes import reference_scores             # FCC / HCP / COL / RND reference lines
# from analysis.scoring import global_order_1cell            # orientation-invariant H1 readout (slow, opt-in)
# from analysis.prototypes import global_order_reference

In [ ]:
# arena_3d.py ships Arena3DExperiment but no Arena3DConfig, so define one here.
# WHY a separate config: exp.save() writes to runs/{run_name}.npz. If we reused Arena2DConfig
# with the same n_steps / kappa / seed, the 3-D run would collide with and OVERWRITE the 2-D
# baseline .npz. A distinct run_name keeps the two experiments side by side.
class Arena3DConfig(ExperimentConfig):
    @property
    def run_name(self) -> str:
        return f"arena3d_T{self.n_steps}_kap{self.kappa}_seed{self.seed}"

g_vec = np.array([0., 0., -9.81])                             # gravity still defines "down" for the filter
cfg = RunConfig(
    network=NetworkConfig(build_connectivity=False),          # validated defaults, torch FFT backend
    experiment=Arena3DConfig(n_steps=250000),
)

# record=True so the saved .npz contains S_tot_buffer, which the 3-D scorer reads.
# (Notebook 4 used record=False because score_2d_from_map scored the in-memory ratemap;
#  score_run reads S_tot_buffer from disk instead, and base.save only writes it when recording.)
# plane_mode="bayesian" is the H1/H2 condition. Flip to "true" for the control arm of the 3x2 design.
exp = Arena3DExperiment(cfg, record=True, plane_mode="bayesian")

In [ ]:
scale = cfg.experiment.scale
print("scale:", scale, " speed:", cfg.experiment.speed,
      " target_speed_rad:", getattr(cfg, "target_speed_rad", None))
print("commanded theta_dot = scale*speed =", scale * cfg.experiment.speed, "rad/step")
print("ceiling = 0.0168  -> saturated?", scale * cfg.experiment.speed > 0.0168)

In [ ]:
print("Running ...")
t0 = time.time()
result = exp.run_experiment(g=g_vec)
exp.save(result, f"workspace/runs/{cfg.experiment.run_name}.npz")
print(f"Saved runs/{cfg.experiment.run_name}.npz  ({time.time()-t0:.1f}s)")

world_pos  = result.world_pos
torus_gt   = result.torus_gt
theta_hist = result.theta_hist
gap        = result.gap_hist
n_hat_hist = result.n_hat_hist
T          = cfg.experiment.n_steps

print(f"Filter gap  t=100: {gap[99]:.4f}  t=500: {gap[499]:.4f}  t={T}: {gap[-1]:.4f}")
print(f"n_hat at t={T}: {n_hat_hist[-1]}  (target [0, 0, 1] if the motion-reference plane stays horizontal)")

# theta_3 diagnostic FLIPS relative to notebook 4. On the flat floor theta_3 == 0; here theta_3
# encodes HEIGHT (the columnar third axis), so a WIDE decoded range is expected and correct, not a bug.
print(f"theta_3 decoded range: [{theta_hist[:, 2].min():.4f}, {theta_hist[:, 2].max():.4f}]  "
      f"(should span a range now; z is encoded on theta_3)")
print(f"MADE error  full: {result.mean_norm_error:.4f}  after t=200: {result.norm_error[200:].mean():.4f}")

In [ ]:
fig = plt.figure(figsize=(5.5, 5))
ax = fig.add_subplot(111, projection="3d")
ax.plot(world_pos[:, 0], world_pos[:, 1], world_pos[:, 2], lw=0.4, alpha=0.6, color="steelblue")
ax.scatter(*world_pos[0],  color="green", s=60, label="start")
ax.scatter(*world_pos[-1], color="red",   s=60, label="end")
ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)"); ax.set_zlabel("z (m)")
ax.set_title("Physical world trajectory (volumetric 3D arena)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, _ = visualize_trajectory_projections(
    torus_gt, theta_hist,
    title="Volumetric 3D arena, ground truth vs decoded on the torus manifold",
)
plt.show()

In [ ]:
fig, _ = plot_pi_error(torus_gt, theta_hist)
plt.show()

## 3-D structure scoring (replaces the 2-D hexagonal gridness)

`score_run` is the volumetric pipeline: rate volume → 3-D autocorrelation → structure scores
χ_FCC / χ_HCP / χ_COL (Gong & Yu, 2021; Grieves et al., 2021), plus the modified radial
autocorrelation (MRA) and inter-field distances. The prototype reference lines (FCC, HCP, COL,
RND) from `prototypes.py` are what make a raw χ interpretable.

**H1 prediction:** the real χ overlaps the **RND** reference (no global lattice), while local order
appears in MRA / first-ring / IFD. Do **not** read a low χ_FCC / χ_HCP as the CAN failing — that
is the predicted volumetric signature.

In [ ]:
npz_path = f"workspace/runs/{cfg.experiment.run_name}.npz"
si  = scoring_input_from_npz(npz_path)            # -> positions in [-1,1]^3, per-neuron-normalised activity
res = score_run(si)                                # full 3-D pipeline: chi_FCC/HCP/COL, MRA, IFD, info, sparsity
ref = reference_scores(n_repeats=30)               # prototype lines: FCC / HCP / COL / RND

print("Real-run structure scores:")
print(res)
print()
print("Prototype reference scores (compare real chi against these):")
print(ref)
# H1 check: real chi_FCC / chi_HCP should land near the RND line, not the FCC/HCP lines.
# Local order (if present) shows up in MRA / inter-field distance, not in the global chi.

## Volumetric field view

Notebook 4 plotted fields on the **floor** (an XY projection). In a volume that projection is
exactly the view Grieves et al. (2021) showed is near chance, so it is only a sanity glance here,
not the H1 test. Below we instead scatter the **3-D rate volume** for one cell.

In [ ]:
# Build the 3-D rate volume for one cell from the same ScoringInput and scatter the high-rate voxels.
# activity_rate_map(si, dims=3) per the technical reference -> array of shape (bins, bins, bins, n_cells).
cell = 0
vol  = activity_rate_map(si, dims=3)               # confirm signature against scoring.py
m    = np.nan_to_num(np.asarray(vol)[..., cell], nan=0.0)

bins = m.shape[0]
c    = np.linspace(-1, 1, bins)
X, Y, Z = np.meshgrid(c, c, c, indexing="ij")

thr  = m > np.percentile(m[m > 0], 80) if np.any(m > 0) else m > 0   # top ~20% of active voxels
fig  = plt.figure(figsize=(5, 4.5))
ax   = fig.add_subplot(111, projection="3d")
p = ax.scatter(X[thr], Y[thr], Z[thr], c=m[thr], cmap="Reds", s=12, alpha=0.7)
ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_zlim(-1, 1)
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
ax.set_title(f"Cell {cell}: 3-D firing-rate volume (top 20% voxels)")
fig.colorbar(p, ax=ax, shrink=0.6, label="rate")
ax.view_init(elev=22, azim=-60)
plt.tight_layout()
plt.show()